In [1]:
!git clone https://github.com/jburroni/IntroPPLs26.git

%cd IntroPPLs26/notebooks/Jun-24

Cloning into 'IntroPPLs26'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 196 (delta 70), reused 106 (delta 39), pack-reused 58 (from 1)
Receiving objects: 100% (196/196), 40.16 MiB | 16.36 MiB/s, done.
Resolving deltas: 100% (82/82), done.
/content/IntroPPLs26/notebooks/Jun-24


# Activity 4 - Sequential Monte Carlo on a stack machine

Single-site MH keeps **one** execution and edits it, re-running the whole trace each step. On a long sequence the chain crawls. SMC instead keeps **many** executions alive, advances them together, and resamples. That needs an evaluator that can **pause** an execution at an `observe` and **resume** it later.

We write the evaluator as an explicit **stack machine**. The control stack *is* the continuation, reified as data, so pausing is just returning from the loop and `fork` (for resampling) is copying the stacks.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from minippl import parse_one, PRIMITIVES, Symbol

## A sequential model in the s-expr language

A 1-D random walk with one noisy reading per step, written as a flat `let` (the evaluator runs the bindings in order, so the `observe`s fire in sequence). We build the program string for any length `T` and simulate data from it.

In [5]:
def random_walk_model(ys, trans=1.0, obs=0.5):
    """(let [z1 .. o1 (observe ..) z2 .. o2 .. ] zT) for the given readings."""
    lines = ['(let [', f'      z1 (sample (normal 0 1))', f'      o1 (observe (normal z1 {obs}) {ys[0]!r})']
    for t in range(2, len(ys) + 1):
        lines.append(f'      z{t} (sample (normal z{t-1} {trans}))')
        lines.append(f'      o{t} (observe (normal z{t} {obs}) {ys[t-1]!r})')
    lines.append(f'     ] z{len(ys)})')
    return '\n'.join(lines)

rng0 = np.random.default_rng(7)
T = 30
z_true = np.cumsum(rng0.normal(0, 1.0, size=T))
ys = [float(v) for v in np.round(z_true + rng0.normal(0, 0.5, size=T), 4)]   # plain floats: clean literals
model = random_walk_model(ys)
print(model[:160], '...')

(let [
      z1 (sample (normal 0 1))
      o1 (observe (normal z1 0.5) -0.7638)
      z2 (sample (normal z1 1.0))
      o2 (observe (normal z2 0.5) 0.0611)
    ...


## The SMC interpreter: an explicit stack machine

`resume(m)` steps until the next `observe`, then returns: the leftover control stack `C` and value stack `V` already are "the rest of the computation," so there is nothing else to capture. `m.fork` duplicates a paused execution and gives the child its own rng, so duplicates diverge on their next `sample`.

In [ ]:
class M:
    """A suspendable execution: control stack C, value stack V, plus rng and weight."""
    __slots__ = ('C', 'V', 'rng', 'log_w')
    def __init__(self, program, rng):
        self.C = [('ev', parse_one(program), {})]; self.V = []
        self.rng, self.log_w = rng, 0.0
    def fork(self, rng):                                          # duplicate this paused execution
        c = M.__new__(M); c.C, c.V = list(self.C), list(self.V)   # copy the two stacks: the rest of the computation
        c.rng, c.log_w = rng, self.log_w                          # a fresh rng so the copies diverge
        return c

def _push_body(C, body, env):
    C.append(('ev', body[-1], env))                       # value of the let is the last body expr
    for b in reversed(body[:-1]):
        C.append(('discard',)); C.append(('ev', b, env))  # earlier exprs run for effect

def resume(m):
    """Advance until an observe pauses the machine, or the program finishes."""
    C, V = m.C, m.V
    while C:
        instr = C.pop(); t = instr[0]
        if t == 'ev':
            _, e, env = instr
            if type(e) is Symbol:          V.append(env[e])
            elif not isinstance(e, list):  V.append(e)
            else:
                op = e[0]
                if   op == 'let':     C.append(('letk', e[1], 0, e[2:], env));  C.append(('ev', e[1][1], env))
                elif op == 'if':      C.append(('ifk', e[2], e[3], env));       C.append(('ev', e[1], env))
                elif op == 'sample':  C.append(('samplek',));                   C.append(('ev', e[1], env))
                elif op == 'observe': C.append(('observek',)); C.append(('ev', e[2], env)); C.append(('ev', e[1], env))
                else:
                    C.append(('primk', op, len(e) - 1))
                    for a in reversed(e[1:]): C.append(('ev', a, env))
        elif t == 'letk':
            _, binds, i, body, env = instr
            env = dict(env); env[binds[2*i]] = V.pop()
            if 2*(i+1) < len(binds): C.append(('letk', binds, i+1, body, env)); C.append(('ev', binds[2*(i+1)+1], env))
            else:                    _push_body(C, body, env)
        elif t == 'ifk':      _, th, el, env = instr; C.append(('ev', th if V.pop() else el, env))
        elif t == 'discard':  V.pop()
        elif t == 'samplek':  d = V.pop(); V.append(d.sample(m.rng))           # draw a new latent from the prior
        elif t == 'observek':                                                  # Q4: weight by this reading, then pause
            ...
            return ('paused', lp)                                              # pausing is returning; C and V hold the rest
        elif t == 'primk':
            _, op, n = instr; a = [V.pop() for _ in range(n)][::-1]; V.append(PRIMITIVES[op](*a))
    return ('done', V[-1])

## The SMC driver

Resume every particle to its next `observe`, weight by that one reading, resample (duplicate each survivor with `m.fork`), and repeat until every particle finishes. The product of average incremental weights is an unbiased estimate of the evidence `p(y)`.

In [ ]:
def smc(program, L=4000, seed=0):
    g = np.random.default_rng(seed)
    ms = [M(program, np.random.default_rng(int(s))) for s in g.integers(1, 2**62, L)]
    logZ = 0.0
    while True:
        outs = [resume(m) for m in ms]                         # advance each particle to its next observe
        if outs[0][0] == 'done':                               # all programs finished together
            finals = np.array([float(o[1]) for o in outs])     # the final states X_T
            return finals, logZ
        inc = np.array([o[1] for o in outs])                   # incremental weight = this observe's log-prob
        mx = inc.max(); w = np.exp(inc - mx); W = w / w.sum()
        logZ += mx + np.log(w.mean())                          # bank the evidence factor
        idx = g.choice(L, size=L, p=W); cs = g.integers(1, 2**62, L)
        ms = [ms[a].fork(np.random.default_rng(int(cs[k]))) for k, a in enumerate(idx)]

smc_zT, smc_logZ = smc(model, L=4000, seed=0)
print('SMC  E[z_T] =', round(float(smc_zT.mean()), 3), '   log Z =', round(smc_logZ, 3))

## Contrast: single-site MH on the same model

This is the addressed, **run-to-completion** evaluator: each proposal re-runs the whole trace, reusing the other sites by address. It works, but on a long correlated sequence the chain mixes slowly.

In [ ]:
class State:
    def __init__(self, rng, x0=None, cache=None):
        self.rng, self.x0, self.cache = rng, x0, cache or {}
        self.X, self.logP_s, self.logP_o, self.log_w = {}, {}, {}, 0.0

def evaluate(e, env, st, a=()):
    if isinstance(e, Symbol): return env[e]
    if not isinstance(e, list): return e
    op, *args = e
    if op == 'let':
        binds, *body = args; env = dict(env)
        for k in range(0, len(binds), 2): env[binds[k]] = evaluate(binds[k+1], env, st, a + ('let', k))
        out = None
        for n, b in enumerate(body): out = evaluate(b, env, st, a + ('body', n))
        return out
    if op == 'if':
        test, then, els = args
        return evaluate(then, env, st, a+('then',)) if evaluate(test, env, st, a+('test',)) else evaluate(els, env, st, a+('else',))
    if op == 'sample':
        d = evaluate(args[0], env, st, a + ('d',))
        x = st.cache[a] if (a != st.x0 and a in st.cache) else d.sample(st.rng)
        st.X[a] = x; st.logP_s[a] = d.log_prob(x); return x
    if op == 'observe':
        d = evaluate(args[0], env, st, a + ('d',)); v = evaluate(args[1], env, st, a + ('v',))
        st.logP_o[a] = d.log_prob(v); st.log_w += st.logP_o[a]; return v
    fn = PRIMITIVES[op] if op in PRIMITIVES else evaluate(op, env, st, a + ('fn',))
    return fn(*[evaluate(arg, env, st, a + (i,)) for i, arg in enumerate(args)])

def run(program, rng, x0=None, cache=None):
    st = State(rng, x0=x0, cache=cache); value = evaluate(parse_one(program), {}, st); return value, st

def single_site_mh(program, rng, S=20000):
    x, st = run(program, rng); chain, acc = [float(x)], 0
    for _ in range(S):
        a0 = list(st.X)[int(rng.integers(len(st.X)))]          # pick a sample site
        x2, st2 = run(program, rng, x0=a0, cache=st.X)         # resample it, reuse the rest by address
        n_old, n_new = len(st.X), len(st2.X)
        fwd = {a0} | (set(st2.X) - set(st.X)); rev = {a0} | (set(st.X) - set(st2.X))
        num = sum(p for k, p in st2.logP_s.items() if k not in fwd) + sum(st2.logP_o.values())
        den = sum(p for k, p in st.logP_s.items()  if k not in rev) + sum(st.logP_o.values())
        log_alpha = (np.log(n_old) - np.log(n_new)) + (num - den)
        if np.log(rng.random()) < log_alpha: x, st, acc = x2, st2, acc + 1
        chain.append(float(x))
    return np.array(chain), acc / S

ssmh_chain, acc = single_site_mh(model, np.random.default_rng(1), S=20000)
print('SSMH E[z_T] =', round(float(ssmh_chain.mean()), 3), '   accept =', round(acc, 2))

## Score both against the exact answer

The model is linear-Gaussian, so a Kalman filter gives the exact `E[z_T | y]`. We also measure the single-site chain's **effective sample size**: how many independent draws its correlated samples are worth.

In [ ]:
def kalman_last(ys, trans=1.0, obs=0.5):
    q, r = trans**2, obs**2; m, P = 0.0, 1.0
    for i, y in enumerate(ys):
        if i > 0: P += q
        K = P / (P + r); m += K * (y - m); P *= (1 - K)
    return m

def ess_chain(x):
    x = x - x.mean(); n = len(x); v = np.dot(x, x) / n
    if v == 0: return float(n)
    s, k = 0.0, 1
    while k < n:
        c = np.dot(x[:-k], x[k:]) / (n - k)
        ac = c / v
        if ac <= 0: break
        s += ac; k += 1
    return n / (1 + 2 * s)

exact = kalman_last(ys)
print(f'exact  E[z_T|y] (Kalman) : {exact:7.3f}')
print(f'SMC    E[z_T|y]   (stack)  : {smc_zT.mean():7.3f}   |err| = {abs(smc_zT.mean()-exact):.3f}')
print(f'SSMH   E[z_T|y]   (chain)  : {ssmh_chain.mean():7.3f}   |err| = {abs(ssmh_chain.mean()-exact):.3f}')
print(f'SSMH effective samples   : {ess_chain(ssmh_chain):.0f} of {len(ssmh_chain)}  (slow mixing on the long chain)')

## The picture

Left: the single-site chain wanders slowly around `z_T`, so thousands of steps are worth a handful of independent draws. Right: SMC's surviving particles sit tight on the exact value.

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(10, 3.4))
axL.plot(ssmh_chain[::5], lw=0.6); axL.axhline(exact, color='k', ls='--', label='exact')
axL.set_title('single-site MH: trace of z_T'); axL.set_xlabel('iteration / 5'); axL.legend()
axR.hist(smc_zT, bins=40, density=True); axR.axvline(exact, color='k', ls='--', label='exact')
axR.set_title('SMC: surviving particles for z_T'); axR.set_xlabel('z_T'); axR.legend()
plt.tight_layout(); plt.show()

## Takeaways

- The explicit stack machine makes execution **resumable** with no coroutines: pausing is returning from the loop, resuming is calling it again, and `fork` is copying the stacks.
- Because it never replays, SMC costs `O(L * T)` and needs no addressing; addresses were only there to make single-site MH's re-runs line up.
- On a long sequence SMC tracks the exact posterior while single-site MH mixes slowly: same compute, very different effective sample size.